In [1]:
import os

def show_inputs(depth=4, max_files_per_dir=6):
    """Print what is attached under /kaggle/input - check it before trusting the rest."""
    root0 = os.environ.get("SAT_INPUT", "/kaggle/input")
    for root, dirs, files in os.walk(root0):
        dirs[:] = sorted(d for d in dirs if not d.startswith("."))
        level = root.replace(root0, "").count(os.sep)
        if level > depth:
            dirs[:] = []
            continue
        print("  " * level + root + "/")
        for fn in sorted(files)[:max_files_per_dir]:
            print("  " * level + "  " + fn)
        if len(files) > max_files_per_dir:
            print("  " * level + f"  ... +{len(files) - max_files_per_dir} more files")

show_inputs()

/kaggle/input/
  /kaggle/input/datasets/
    /kaggle/input/datasets/rifat4018/
      /kaggle/input/datasets/rifat4018/sat-ext/
        /kaggle/input/datasets/rifat4018/sat-ext/sat_ext/
          README.md
          audit.py
          config.py
          dataset.py
          infer.py
          metrics.py
          ... +7 more files
      /kaggle/input/datasets/rifat4018/satelite-coco/
        README.dataset.txt
        README.roboflow.txt
        /kaggle/input/datasets/rifat4018/satelite-coco/test/
          1011_jpeg.rf.141ad0f26ab02f1bfee50847febecb85.jpg
          1016_jpeg.rf.0aef8e3664a12b1681da06035362ed54.jpg
          1025_jpeg.rf.ce6b95f0170aab49e8aab8728fca22a8.jpg
          1067_jpeg.rf.b20f17d8c4416f8b8ed687b14a48966a.jpg
          1077_jpeg.rf.0e1e2f1b96f4487629a56a6b93755174.jpg
          1087_jpeg.rf.4324835fa22676557aae5101128ff1ca.jpg
          ... +50 more files
        /kaggle/input/datasets/rifat4018/satelite-coco/train/
          1001_jpeg.rf.056f37938104493a480e839

In [2]:
import os, sys, warnings
warnings.filterwarnings("ignore")
INPUT_ROOT = os.environ.get("SAT_INPUT", "/kaggle/input")

def _find_code_root():
    for root, dirs, files in os.walk(INPUT_ROOT):
        if {"config.py", "audit.py", "metrics.py", "infer.py"} <= set(files):
            return root
    return None

CODE_ROOT = os.environ.get("SAT_CODE") or _find_code_root()
assert CODE_ROOT, "sat_ext code folder not found - Add Input -> your 'sat-ext' code Dataset"
sys.path.insert(0, CODE_ROOT)
print("CODE_ROOT:", CODE_ROOT)

import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 70)
import config as cfg, utils
utils.ensure_dirs(); utils.set_seed(cfg.SEED)
print("work dir:", cfg.WORK_DIR, "| smoke test:", cfg.SMOKE)

CODE_ROOT: /kaggle/input/datasets/rifat4018/sat-ext/sat_ext
work dir: /kaggle/working | smoke test: False


In [3]:
DATASET = "sky"          # preset: which attached export this notebook audits
PREFIX = cfg.DATASETS[DATASET]["prefix"]      # split names: PREFIX + "orig" / PREFIX + "grouped"
USE_ORB = True        # partial-overlap check between tiles; set False only if you are short of time
utils.warn_if_gpu_wasted()
import dataset as ds, audit, split
AUDIT = os.path.join(cfg.AUDIT_OUT, DATASET)
os.makedirs(AUDIT, exist_ok=True)
RAW_ROOT = ds.find_coco_root(DATASET)
print("raw export:", RAW_ROOT)
raw = ds.load_raw(RAW_ROOT)
cmap = ds.category_map(raw)
for s, c in raw.items():
    print(f"{s:<5} categories {[(x['id'], x['name']) for x in c['categories']]}  ->  used as YOLO classes {cmap[s]}")
imgs = ds.image_table(raw)
print("images whose file is missing:", int((~imgs.exists).sum()))

raw export: /kaggle/input/datasets/rifat4018/skysealand-v14
train categories [(0, 'objects'), (1, 'airplane'), (2, 'boat'), (3, 'car'), (4, 'ship')]  ->  used as YOLO classes {1: 0, 2: 1, 3: 2, 4: 1}
val   categories [(0, 'objects'), (1, 'airplane'), (2, 'boat'), (3, 'car'), (4, 'ship')]  ->  used as YOLO classes {1: 0, 2: 1, 3: 2, 4: 1}
test  categories [(0, 'objects'), (1, 'airplane'), (2, 'boat'), (3, 'car'), (4, 'ship')]  ->  used as YOLO classes {1: 0, 2: 1, 3: 2, 4: 1}
images whose file is missing: 0


## 1. Counts per split (raw export, before any cleaning)
`copies_per_source` = how many images share one Roboflow source name, e.g. `{3: 435}` means 435 sources with 3 copies each.

In [4]:
counts, cats = audit.split_counts(raw, imgs, cmap)
print(counts.set_index("split").T.to_string())
counts.to_csv(f"{AUDIT}/01_split_counts.csv", index=False)

split                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

In [5]:
# reconcile with the numbers printed in the rejected manuscript (paper dataset only)
PAPER = {"train images": 1305, "val images": 54, "test images": 55,
         "train boxes": 44909, "val boxes": 2032, "test boxes": 1826,
         "train negative x": 358, "train negative y": 366}
c = counts.set_index("split")
MEAS = {"train images": c.loc["train", "images"], "val images": c.loc["val", "images"], "test images": c.loc["test", "images"],
        "train boxes": c.loc["train", "annotations"], "val boxes": c.loc["val", "annotations"], "test boxes": c.loc["test", "annotations"],
        "train negative x": c.loc["train", "neg_x"], "train negative y": c.loc["train", "neg_y"]}
rec = pd.DataFrame({"paper": PAPER, "measured": MEAS})
rec["match"] = rec.paper == rec.measured
if DATASET == "main":
    print(rec.to_string())
n_src = imgs.source_id.nunique()
print(f"\nunique source images over all splits: {n_src}  (the paper's '1,414 base images' counts augmented copies)")
rec.to_csv(f"{AUDIT}/01_paper_reconciliation.csv")


unique source images over all splits: 1307  (the paper's '1,414 base images' counts augmented copies)


## 2. Box cleaning (logged)

In [6]:
boxes, clean_log = ds.clean_annotations(raw, cmap)
print(clean_log.groupby(["split", "action"]).size().unstack(fill_value=0).to_string())
clean_log.to_csv(f"{AUDIT}/02_clean_log.csv", index=False)
print("boxes kept:", sum(len(v) for v in boxes.values()))

action  clipped  degenerate_removed
split                              
test          3                   0
train        32                   1
val           4                   0
boxes kept: 19102


## 3. Exact and near duplicates (SHA-256, pHash under the 8 flips/rotations)

In [7]:
with utils.timer("hashes"):
    feat, PH = audit.image_features(imgs)
sha_dup = feat[feat.duplicated("sha256", keep=False)].sort_values("sha256")
print("byte-identical images:", len(sha_dup))
ph = audit.phash_pairs(feat, PH)
ph.to_csv(f"{AUDIT}/03_phash_pairs.csv", index=False)
print(f"pHash pairs (distance <= {cfg.PHASH_MAX_HAMMING}): {len(ph)}; across splits: {int(ph.cross_split.sum()) if len(ph) else 0}; "
      f"across DIFFERENT source names: {int((~ph.same_source).sum()) if len(ph) else 0}")
if len(ph):
    print(ph.sort_values("hamming").head(20)[["key_i", "key_j", "hamming", "cross_split", "same_source"]].to_string(index=False))
    cross = ph[ph.cross_split | ~ph.same_source].sort_values("hamming")
    p = audit.pair_montage(feat, cross, f"{AUDIT}/03_phash_pairs_crosssplit.png",
                           "pHash near-duplicates across splits / across source names - CHECK BY EYE")
    print("montage:", p)

[timer] hashes: 1.3 min (78s)
byte-identical images: 0
pHash pairs (distance <= 8): 2; across splits: 1; across DIFFERENT source names: 2
                                                         key_i                                                          key_j  hamming  cross_split  same_source
 test/sample_10180_jpg.rf.704a41a5b1384a3e980257248f113bd5.jpg train/sample_10106_jpg.rf.eddacd00ab5fd2640a116a5e26f02dc8.jpg        8         True        False
train/sample_10130_jpg.rf.8271ac652113a433b3928d192e76e7d8.jpg train/sample_10131_jpg.rf.679e76304fed997b55a399843f8ca904.jpg        8        False        False
montage: /kaggle/working/audit/sky/03_phash_pairs_crosssplit.png


## 4. Overlapping tiles (ORB features + RANSAC homography)
One representative per source image (the least-rotated copy). Pairs with at least `ORB_MIN_INLIERS` geometric inliers are
treated as the same ground area. **Look at the montage**: if unrelated scenes appear (repetitive parking lots can fool
the matcher), raise `cfg.ORB_MIN_INLIERS` in this cell and rerun this cell and the ones below.

In [8]:
# cfg.ORB_MIN_INLIERS = 40     # <- raise only if the montage shows unrelated scenes
rep = audit.representatives(feat)
orb = None
if USE_ORB:
    with utils.timer("ORB overlap"):
        orb = audit.orb_pairs(feat.path.values[rep])
    orb["key_i"], orb["key_j"] = feat.key.values[rep[orb.i]], feat.key.values[rep[orb.j]]
    orb["split_i"], orb["split_j"] = feat.src_split.values[rep[orb.i]], feat.src_split.values[rep[orb.j]]
    orb.to_csv(f"{AUDIT}/04_orb_pairs.csv", index=False)
    strong = orb[orb.inliers >= cfg.ORB_MIN_INLIERS]
    print(f"overlapping pairs: {len(strong)} (across splits: {int((strong.split_i != strong.split_j).sum())})")
    print("inlier distribution of the 30 strongest pairs:", orb.inliers.head(30).tolist())
    mont = orb.assign(i=rep[orb.i], j=rep[orb.j])
    audit.pair_montage(feat, mont[mont.inliers >= cfg.ORB_MIN_INLIERS], f"{AUDIT}/04_orb_overlap_top.png",
                       "ORB overlap - strongest pairs", score_col="inliers")
    border = mont[(mont.inliers >= cfg.ORB_MIN_INLIERS * 0.6) & (mont.inliers < cfg.ORB_MIN_INLIERS * 1.6)]
    audit.pair_montage(feat, border, f"{AUDIT}/04_orb_overlap_borderline.png",
                       f"ORB pairs near the threshold ({cfg.ORB_MIN_INLIERS}) - are these the same place?", score_col="inliers")

ORB: 1307 representative images, 853,471 pairs, 4 workers
[timer] ORB overlap: 31.7 min (1904s)
overlapping pairs: 244 (across splits: 90)
inlier distribution of the 30 strongest pairs: [287, 235, 232, 230, 215, 215, 215, 215, 214, 209, 208, 206, 202, 198, 197, 191, 191, 182, 172, 171, 167, 166, 164, 161, 159, 150, 150, 146, 142, 141]


## 5. Groups and leakage of the original split

In [9]:
feat_g, evidence = audit.build_groups(feat, ph, orb, rep, use_orb=USE_ORB)
leak = audit.leakage_summary(feat_g, ph, orb, rep)
leak["evidence_links"] = evidence
utils.save_json(leak, f"{AUDIT}/05_leakage_summary.json")
feat_g.drop(columns=["path"]).to_csv(f"{AUDIT}/05_images_groups.csv", index=False)
for k, v in leak.items():
    print(f"  {k:<45} {v}")
if leak["largest_group_share_of_sources"] > 0.15:
    print("\nWARNING: one group holds more than 15% of all source images. Check 04_orb_overlap_borderline.png;"
          " if those pairs are NOT the same place, raise cfg.ORB_MIN_INLIERS (cell above) and rerun from there.")

  groups_total                                  1108
  groups_spanning_splits                        50
  images_in_spanning_groups                     168
  source_ids_in_more_than_one_split             0
  largest_groups_sources                        [23, 10, 9, 7, 7]
  largest_group_share_of_sources                0.017597551644988524
  phash_pairs_cross_split                       1
  orb_overlap_pairs                             244
  orb_overlap_pairs_cross_split                 90
  val_images_whose_group_has_train_images       23
  test_images_whose_group_has_train_images      32
  evidence_links                                {'phash': 2, 'orb': 244}


## 6. Object sizes (areas in px² on the 640×640 images) and ground-truth samples

In [10]:
sizes = audit.box_sizes(boxes, imgs)
tab = sizes.groupby(["split", "bucket"]).size().unstack(fill_value=0).reindex(columns=[b for b, _, _ in cfg.BUCKETS], fill_value=0)
print(tab.to_string()); print((tab.T / tab.sum(1)).T.round(3).to_string())
tab.to_csv(f"{AUDIT}/06_size_buckets.csv")
print("COCO convention (paper): small = area < 32^2 px^2, medium = 32^2..96^2, large >= 96^2; micro (ours) = area < 16^2")
print(audit.size_figure(sizes, f"{AUDIT}/06_size_distribution.png"))
for s in ("train", "val", "test"):
    print(audit.gt_grid(feat, boxes, f"{AUDIT}/06_gt_samples_{s}.png", s))

bucket  micro  tiny16_32  medium  large
split                                  
test        8        265    1508    294
train     126       1845   10854   2210
val        14        261    1390    327
bucket  micro  tiny16_32  medium  large
split                                  
test    0.004      0.128   0.727  0.142
train   0.008      0.123   0.722  0.147
val     0.007      0.131   0.698  0.164
COCO convention (paper): small = area < 32^2 px^2, medium = 32^2..96^2, large >= 96^2; micro (ours) = area < 16^2
/kaggle/working/audit/sky/06_size_distribution.png
/kaggle/working/audit/sky/06_gt_samples_train.png
/kaggle/working/audit/sky/06_gt_samples_val.png
/kaggle/working/audit/sky/06_gt_samples_test.png


## 7. Write the two splits

In [11]:
man_o = split.orig_manifest(feat_g)
meta_o = ds.write_split(PREFIX + "orig", man_o, imgs, boxes, log=clean_log, extra=dict(dataset=DATASET, leakage=leak))
man_g, ginfo = split.grouped_manifest(feat_g, boxes)
split.check_no_group_leak(man_g)
meta_g = ds.write_split(PREFIX + "grouped", man_g, imgs, boxes, log=clean_log, extra=dict(dataset=DATASET, leakage=leak, grouping=ginfo))
import shutil; shutil.copytree(AUDIT, os.path.join(cfg.SPLITS_OUT, "audit_" + DATASET), dirs_exist_ok=True)

split 'sky_orig' fingerprint bbc7d1160d57
       images  groups  annotations  empty_images  plane  vehicle  ship
train    1048     907        15035             0   3927     5454  5654
val       132     129         1992             0    367      679   946
test      127     125         2075             0    553      798   724
grouped split: 1108 groups -> test 166, val 166, train 776 (balance score 0.0056; dropped copies 0)
split 'sky_grouped' fingerprint 0b03f1f732a3
       images  groups  annotations  empty_images  plane  vehicle  ship
train     922     776        13472             0   3385     4936  5151
val       193     166         2835             0    726      995  1114
test      192     166         2795             0    736     1000  1059


'/kaggle/working/splits/audit_sky'

## 8. Read this before the GPU notebooks
* `orig` fingerprint and `grouped` fingerprint are printed above. **Every training notebook prints the fingerprint it used;
  they must all be the same** (the aggregation notebook checks it).
* Save Version -> Output -> New Dataset `sat-splits`. Share it with the other accounts (Dataset -> Settings -> Sharing -> add
  the usernames, or make it public if the image licence allows).

In [12]:
print("orig    :", meta_o["fingerprint"], meta_o["summary"])
print("grouped :", meta_g["fingerprint"], meta_g["summary"])
print("files:", sorted(os.listdir(cfg.SPLITS_OUT)))

orig    : bbc7d1160d57 {'train': {'images': 1048, 'groups': 907, 'annotations': 15035, 'empty_images': 0, 'plane': 3927, 'vehicle': 5454, 'ship': 5654}, 'val': {'images': 132, 'groups': 129, 'annotations': 1992, 'empty_images': 0, 'plane': 367, 'ship': 946, 'vehicle': 679}, 'test': {'images': 127, 'groups': 125, 'annotations': 2075, 'empty_images': 0, 'plane': 553, 'vehicle': 798, 'ship': 724}}
grouped : 0b03f1f732a3 {'train': {'images': 922, 'groups': 776, 'annotations': 13472, 'empty_images': 0, 'plane': 3385, 'vehicle': 4936, 'ship': 5151}, 'val': {'images': 193, 'groups': 166, 'annotations': 2835, 'empty_images': 0, 'plane': 726, 'ship': 1114, 'vehicle': 995}, 'test': {'images': 192, 'groups': 166, 'annotations': 2795, 'empty_images': 0, 'plane': 736, 'vehicle': 1000, 'ship': 1059}}
files: ['audit_sky', 'sky_grouped', 'sky_orig']
